In [1]:
!pip install -q datasets vaderSentiment scikit-learn

In [2]:
from datasets import load_dataset
import pandas as pd, html, re
from sklearn.metrics import classification_report, confusion_matrix, f1_score

sent = load_dataset("cardiffnlp/tweet_eval", "sentiment")
test = sent["test"].to_pandas()
names = ["negative", "neutral", "positive"]

def fix_text(t):
    t = re.sub(r"\\u([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), t)
    return html.unescape(t)

def report(y_true, y_pred, title):
    print(title)
    print("Macro-F1:", round(f1_score(y_true, y_pred, average="macro"), 3))
    print(classification_report(y_true, y_pred, target_names=names, digits=3))
    print(pd.DataFrame(confusion_matrix(y_true, y_pred),
                       index=[f"true_{n}" for n in names],
                       columns=[f"pred_{n}" for n in names]))

README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

sentiment/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.78MB            

sentiment/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  901kB            

sentiment/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/validation-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  167kB            

sentiment/validation-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

def vader_label(text):
    c = vader.polarity_scores(text)["compound"]
    if c >= 0.05: return 2     # positive
    if c <= -0.05: return 0    # negative
    return 1                   # neutral

test["text_fixed"] = test["text"].apply(fix_text)
test["vader_pred"] = test["text_fixed"].apply(vader_label)
report(test["label"], test["vader_pred"], "VADER on TweetEval test")

VADER on TweetEval test
Macro-F1: 0.529
              precision    recall  f1-score   support

    negative      0.607     0.559     0.582      3972
     neutral      0.675     0.437     0.531      5937
    positive      0.355     0.714     0.474      2375

    accuracy                          0.530     12284
   macro avg      0.546     0.570     0.529     12284
weighted avg      0.591     0.530     0.536     12284

               pred_negative  pred_neutral  pred_positive
true_negative           2222           724           1026
true_neutral            1283          2594           2060
true_positive            156           524           1695


In [4]:
for s in ["Love waking up early in the morning",
          "This is not good at all",
          "Well the week is off to a great start. #fml"]:
    print(s, "->", vader.polarity_scores(s)["compound"])

Love waking up early in the morning -> 0.6369
This is not good at all -> -0.3412
Well the week is off to a great start. #fml -> 0.7351
